# Тензоры и автоматическое дифференцирование

Авторы BelOAI. CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи самостоятельно, затем обратитесь к разделу решений.

## Уроки по этим темам

Материал этого ноутбука можно изучать по отдельным урокам:

- [Тензоры, формы и устройства](../../python/pytorch-tools/tensors/)
- [Автоматическое дифференцирование](../../python/pytorch-tools/autograd/)
- [Dataset и DataLoader](../../python/pytorch-tools/datasets-loaders/)
- [Режимы модели и управление градиентами](../../python/pytorch-tools/model-modes/)


## Чему научимся

Создавать тензоры на CPU (центральном процессоре компьютера), вычислять градиенты и получать мини-пакеты данных. Для понимания нужны NumPy и глава о градиентах. Понятия функций, аргументов и `raise` используются в том же смысле, что в первых главах Python. Полное обучение нейросетей изучается дальше в маршруте ML.

## Тензоры, типы и устройства

Тензор здесь — многомерный массив чисел. `torch.tensor` создаёт его из чисел или вложенных списков. В PyTorch он имеет форму `shape`, тип `dtype` и устройство `device`. Числовой таблице обычно подходит `torch.float32`; для точной проверки производных удобно взять `float64`. Для целых индексов часто нужен `torch.int64`. Параметр `requires_grad=True` указывает, что по этому тензору понадобятся производные. В примере `w` — два веса линейной модели, `X` — два объекта, `y` — их правильные ответы. Оператор `@` умножает матрицу на вектор.

In [ ]:
import torch

X = torch.tensor([[2., 1.], [1., 3.]], dtype=torch.float64, device="cpu")
w = torch.tensor([1., 1.], dtype=torch.float64, requires_grad=True)
y = torch.tensor([5., 4.], dtype=torch.float64)
assert X.shape == (2, 2) and w.shape == (2,)
assert X.device.type == "cpu"
prediction = X @ w
assert prediction.shape == y.shape


Broadcasting работает по знакомым правилам NumPy: формы ответов и предсказаний проверяйте до вычитания. `.to(device)` переносит тензор и возвращает результат; все участники операции должны быть на совместимых устройствах. В наших примерах ускоритель не нужен.

## Граф вычислений и autograd

`requires_grad=True` включает запись операций над параметром для автоматического дифференцирования. Граф связывает входы и промежуточные результаты. Для скалярной ошибки `backward()` применяет правило цепочки и записывает производные в `.grad`.

Скаляр — одно число; ошибка должна быть скалярной для вызова `backward()` без дополнительных аргументов. Метод `mean()` вычисляет среднее, `X.T` транспонирует таблицу, `torch.allclose` сравнивает тензоры с допуском на численные погрешности.

`detach()` даёт тензор без отслеживания градиента, но может разделять память с исходным. Для независимой копии используйте `.detach().clone()`. `.item()` превращает тензор из одного элемента в обычное число — удобно для журнала, но через такое число градиент уже не проходит.

In [ ]:
loss = ((prediction - y) ** 2).mean()
loss.backward()
expected_gradient = 2 / len(y) * X.T @ (X @ w.detach() - y)
assert torch.allclose(w.grad, expected_gradient)
assert torch.allclose(w.grad, torch.tensor([-4., -2.], dtype=torch.float64))
print(loss.item())  # 2.0


У первого объекта знаковая ошибка −2, у второго 0; средний квадрат равен 2. По формуле из главы о градиентах производные по двум весам равны −4 и −2. `backward` вычисляет их, но сам не меняет веса. ## Накопление и отключение градиентов

Повторные `backward` складывают градиенты, а не заменяют их. В обычном шаге обучения сначала обнуляют старые значения, заново вычисляют ошибку и вызывают `backward`. Старый граф после обратного прохода обычно освобождается, поэтому просто повторить `loss.backward()` нельзя.

`torch.no_grad()` временно отключает построение графа. Так можно обновлять параметры или получать проверочные ответы без лишней памяти. Обновление здесь — вычисление нового набора весов `w - 0.01 * w.grad`, где 0.01 задаёт размер шага. Оно записывается в новое имя `updated`, исходный `w` не меняется. В полноценном обучении обновления поручают оптимизатору — объекту, выполняющему шаги изменения параметров; его разберём вместе с нейросетями.

In [ ]:
w.grad = None
fresh_loss = ((X @ w - y) ** 2).mean()
fresh_loss.backward()
assert torch.allclose(w.grad, expected_gradient)
with torch.no_grad():
    updated = w - 0.01 * w.grad
assert not updated.requires_grad


## Dataset, класс и методы

`Dataset` задаёт, как получить отдельный пример; `DataLoader` собирает примеры в мини-пакеты — небольшие группы для одного вычисления. Нужен минимальный синтаксис класса: класс описывает объект с данными и действиями; методы — функции внутри класса, `self` обозначает текущий объект. `__init__` вызывается при создании, `__len__` задаёт длину, `__getitem__` — получение по индексу. Запись `class Measurements(Dataset):` создаёт наш вид набора данных на основе `Dataset`; в нём определяем нужные методы. `self.features` хранит данные именно этого объекта. При вызове `Measurements(X, y)` Python сам передаст создаваемый объект в `self`, а `X` и `y` — в `features` и `targets`.

## Разобранный пример: два объекта, два мини-пакета

При `len(dataset)` Python вызывает `__len__`, при `dataset[0]` — `__getitem__(0)`. Этот метод возвращает пару «признаки, ответ»; цикл `for batch_X, batch_y in loader` распаковывает такие данные уже для пакета объектов. `torch.Generator().manual_seed(17)` создаёт генератор и задаёт ему начальное состояние.

`batch_size=1` задаёт один объект в пакете, `shuffle=True` перемешивает порядок при обходе, `generator` задаёт источник случайности. `num_workers=0` читает данные в текущем процессе, что удобно для небольшого ноутбука. Последний пакет может быть меньше `batch_size`; не зашивайте его размер в форму ответа.

In [ ]:
from torch.utils.data import Dataset, DataLoader

class Measurements(Dataset):
    def __init__(self, features, targets):
        if len(features) != len(targets):
            raise ValueError("Число объектов и ответов должно совпадать")
        self.features = features
        self.targets = targets

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):
        return self.features[index], self.targets[index]

dataset = Measurements(X, y)
generator = torch.Generator().manual_seed(17)
loader = DataLoader(dataset, batch_size=1, shuffle=True, generator=generator, num_workers=0)
seen = 0
for batch_X, batch_y in loader:
    assert batch_X.shape == (1, 2)
    assert batch_y.shape == (1,)
    seen += len(batch_y)
assert seen == 2


В данном наборе `dataset[0]` возвращает признаки `[2, 1]` и ответ 5. После объединения в пакет из одного объекта форма признаков `(1, 2)`, ответов — `(1,)`. Первая ось теперь обозначает объекты пакета.

## Режимы модели

`torch.nn.Linear(2, 1)` — готовый слой: он принимает два признака и выдаёт одно число по формуле линейной модели с весами и свободным членом. Это объект модели с обучаемыми параметрами. `train()` включает режим обучения, `eval()` — режим оценки. Некоторые слои, которые изучим вместе с нейросетями, ведут себя в этих режимах по-разному. `eval()` сам по себе не отключает градиенты; `no_grad()` не переключает режим модели.

Модель вызывается как функция: `model(X)` вычисляет предсказания. Их форма `(2, 1)` содержит лишнюю для сравнения с `y` последнюю ось длины 1. Метод `.squeeze(-1)` убирает именно эту ось и даёт `(2,)`; `-1` обозначает последнюю ось. Свойство `model.training` истинно в режиме обучения и ложно в режиме оценки.

Перед созданием слоя задаём `torch.manual_seed(17)`: начальные веса можно повторить в том же окружении. Проверка формы сама по себе не означает, что модель обучена.

In [ ]:
torch.manual_seed(17)
model = torch.nn.Linear(2, 1).to(dtype=torch.float64, device="cpu")
model.train()
assert model.training
model.eval()
with torch.no_grad():
    output = model(X).squeeze(-1)
assert output.shape == y.shape
assert not output.requires_grad


Случайные веса этого слоя ещё не обучены: значение выхода не оценивает качество решения.

Для справки: [автоматическое дифференцирование PyTorch](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html).

## Задачи

### 1. Базовая

Что записывается в `w.grad` после `backward()`?

<details>
<summary>Подсказка</summary>

Это не обновлённые веса.

</details>



### 2. Базовая

Отключает ли `model.eval()` вычисление градиентов?

<details>
<summary>Подсказка</summary>

Режим модели и запись графа управляются отдельно.

</details>



### 3. Прикладная

Проверьте autograd для скаляра z = 3 и ошибки (z − 1)².

<details>
<summary>Подсказка</summary>

Примените правило производной квадрата и подставьте заданное z.

</details>



### 4. Прикладная

Создайте загрузчик тех же двух объектов с batch_size=3 без перемешивания. Какова форма единственного пакета?

<details>
<summary>Подсказка</summary>

Последний пакет может быть неполным.

</details>



### 5. Олимпиадная

В отдельном опыте для `z = 3` ошибка `(z - 1) ** 2` после первого `backward()` дала градиент 4. Что произойдёт с ним, если заново вычислить ту же ошибку и вызвать `backward()` без обнуления?

<details>
<summary>Подсказка</summary>

Градиенты накапливаются.

</details>



### 6. Олимпиадная

Почему `float(loss)` или `loss.item()` нельзя использовать вместо исходного тензора для последующего backward?

<details>
<summary>Подсказка</summary>

Обычное число не хранит связь с графом.

</details>



## Итоги и самопроверка

Вручную проверьте градиент маленького примера и формы каждого пакета. Объясните различия между `eval`, `no_grad` и обнулением градиентов. Эти навыки понадобятся при обучении нейросетей.


## Решения задач

### Задача 1

Производные скалярной ошибки по компонентам w, накопленные с предыдущими градиентами, если они были.

### Задача 2

Нет. Для отключения записи графа при оценке используйте `torch.no_grad()`.

### Задача 3

In [ ]:
z = torch.tensor(3., dtype=torch.float64, requires_grad=True)
((z - 1) ** 2).backward()
assert torch.allclose(z.grad, torch.tensor(4., dtype=torch.float64))


### Задача 4

In [ ]:
small_loader = DataLoader(dataset, batch_size=3, shuffle=False, num_workers=0)
packets = list(small_loader)
assert len(packets) == 1
assert packets[0][0].shape == (2, 2)
assert packets[0][1].shape == (2,)


### Задача 5

Он станет 8 вместо 4. Перед следующим независимым шагом нужно `z.grad = None` или обнуление через оптимизатор.

### Задача 6

Преобразование разрывает путь автоматического дифференцирования. Для обратного прохода нужен исходный тензор ошибки, число сохраняют лишь для вывода или журнала.